# Defining a custom network

`jVMC_exp.nets` provides several ready-made architectures (RBMs, CNNs, transformers, ...), but any [Flax](https://flax.readthedocs.io) module can be used as a variational wave function. This notebook shows the minimal requirements, using a complex restricted Boltzmann machine (RBM) written from scratch as an example.

The wave function is represented through its logarithm, $\psi_\theta(s) = \exp\big(f_\theta(s)\big)$, where $f_\theta$ is the network. The network only has to:

- take **a single configuration** $s$ (an array of shape `sampleShape`, with entries 0/1 for spins 1/2) and
- return **a scalar**, the log-amplitude $\ln\psi_\theta(s)$.

Batching over many configurations, vectorization and the distribution across devices are handled by `NQS`, so they do not need to be written in the network.

In [1]:
import jax
import jax.numpy as jnp
import flax.linen as nn

import jVMC_exp

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Running in single-node mode (JVMC_USE_DISTRIBUTED not set)


## 1. The network

An RBM with $M$ hidden units has the log-amplitude

$$
\ln\psi_\theta(s) = \sum_{j=1}^{M} \ln\cosh\Big(\sum_i W_{ji}\, \sigma_i + b_j\Big), \qquad \sigma_i = 2 s_i - 1 \in \{-1, 1\}.
$$

In Flax this is a `Dense` layer followed by $\ln\cosh$ and a sum. Two details matter:

- `param_dtype=jVMC_exp.global_defs.DT_PARAMS_CPX` makes the parameters complex. Note that Flax's `dtype` argument only sets the dtype of the computation, not of the parameters.
- `kernel_init` sets how the weights are initialized. Here we use the complex initializer of `jVMC_exp.nets.initializers`.

The hyperparameters (here `numHidden`) are declared as class attributes, as usual for Flax modules.

In [2]:
class MyNet(nn.Module):
    numHidden: int = 2

    @nn.compact
    def __call__(self, s):
        s = 2 * s - 1  # Go from 0/1 representation to -1/1
        h = nn.Dense(
            features=self.numHidden,
            param_dtype=jVMC_exp.global_defs.DT_PARAMS_CPX,
            kernel_init=jVMC_exp.nets.initializers.cplx_init,
        )(s)

        return jnp.sum(jnp.log(jnp.cosh(h)))

## 2. Wrapping it into a variational state

The network is turned into a variational wave function by `NQS`, exactly as for the built-in networks. At construction, `NQS` initializes the parameters and detects whether they are real or complex and whether the network is holomorphic in them. This determines how gradients are computed. A complex RBM is holomorphic.

In [3]:
L = 4  # system size

net = MyNet(numHidden=7)
psi = jVMC_exp.vqs.NQS(net, sampleShape=L, batchSize=2**L, seed=1234)

print("Number of parameters:", psi.numParameters)
print("Real parameters:     ", psi.realParams)
print("Holomorphic:         ", psi.holomorphic)

Number of parameters: 35
Real parameters:      False
Holomorphic:          True


## 3. Evaluating the wave function

Calling `psi` on an array of configurations of shape `(N, *sampleShape)` returns $\ln\psi(s)$ for each of them, in a 1D array of length `N`.

In [4]:
# Create a set of 2**L random input configurations
configs = jax.random.bernoulli(jax.random.PRNGKey(4321), shape=(2**L, L))

# Evaluate the log-amplitudes
log_psi = psi(configs)

print("log_psi.shape:", log_psi.shape)
print(log_psi[:4])

log_psi.shape: (16,)
[-7.38841698e-05-1.45356345e-04j -3.46799911e-04+8.75095277e-04j
 -3.46799911e-04+8.75095277e-04j -2.12515614e-05+8.68297896e-05j]


## Next steps

`psi` can now be used like any other variational state: for example, replace `jVMC_exp.nets.CpxRBM` by `MyNet` in the ground state search example.

A network can optionally define a few more methods, such as `sample(key)` for direct (autoregressive) sampling, or `eval_ratio(s, sp)` to compute $\psi(s')/\psi(s)$ faster in Monte Carlo sampling. See the documentation of `jVMC_exp.vqs.NQS`.